# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Ablation: alpha < 1, rank ratio = 1 || alpha = 1, rank ratio <=1, random projection

## load data

In [6]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_trans'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df.head(4)

,model,seq_len,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,lradj,patience,train_epochs,mse,mae
77,FreTS,96,96,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.171692,0.224669
44,FreTS,96,192,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.212493,0.263629
89,FreTS,96,336,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.261259,0.299690
300,FreTS,96,720,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.335380,0.360984


## load data rebb

In [5]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_trans_rebb'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df_rebb = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    df_rebb.append(result)

df_rebb = pd.concat(df_rebb, ignore_index=True)
df_rebb.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df_rebb.head(4)

,model,seq_len,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,lradj,patience,train_epochs,mse,mae
0,FreTS,96,96,Weather_Random,0.0004,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.169945,0.214964
3,FreTS,96,96,Weather_Random,0.0003,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.171654,0.216642
6,FreTS,96,96,Weather_Random,0.0002,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.175259,0.220235
12,FreTS,96,96,Weather_Random,0.0020,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.168404,0.212037


## preprocess

In [7]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes_all = pd.read_csv(f'{save_root}/finetune_all_results.csv')

base = baselines.copy()
base = base[
    ((base.data_id == 'ETTh1') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ETTm1') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ECL') & (base.model == 'iTransformer')) |
    ((base.data_id == 'Weather') & (base.model == 'FreTS'))
]

aba1 = finetunes_all.copy()
aba1 = aba1[
    (aba1.pca_dim == 'T') &
    (aba1.use_weights == 0) &
    (aba1.auxi_loss == 'MAE') &
    (aba1.reinit == 1) &
    (aba1.seq_len == 96) &
    (aba1.rank_ratio == 1.0)
]
aba1 = aba1[
    ((aba1.data_id == 'ETTh1_PCA') & (aba1.model == 'Fredformer')) |
    ((aba1.data_id == 'ETTm1_PCA') & (aba1.model == 'Fredformer')) |
    ((aba1.data_id == 'ECL_PCA') & (aba1.model == 'iTransformer')) |
    ((aba1.data_id == 'Weather_PCA') & (aba1.model == 'FreTS'))
]
aba1.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba1.drop(columns=['rec_lambda'], inplace=True)


aba2 = df.copy()
aba2 = aba2[
    (aba2.pca_dim == 'T') &
    (aba2.use_weights == 0) &
    (aba2.auxi_loss == 'MAE') &
    (aba2.reinit == 1) &
    (aba2.seq_len == 96) &
    (aba2.rank_ratio < 1.0) &
    (aba2.learning_rate.isin([0.0005]))
]
aba2 = aba2[
    ((aba2.data_id == 'ETTh1_Random') & (aba2.model == 'Fredformer')) |
    ((aba2.data_id == 'ETTm1_Random') & (aba2.model == 'Fredformer')) |
    ((aba2.data_id == 'ECL_Random') & (aba2.model == 'iTransformer')) |
    ((aba2.data_id == 'Weather_Random') & (aba2.model == 'FreTS'))
]
aba2.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba2.drop(columns=['rec_lambda'], inplace=True)


aba21 = df.copy()
aba21 = aba21[
    (aba21.pca_dim == 'T') &
    (aba21.use_weights == 0) &
    (aba21.auxi_loss == 'MAE') &
    (aba21.reinit == 1) &
    (aba21.seq_len == 96) &
    (aba21.rank_ratio == 1.0)
]
aba21 = aba21[
    ((aba21.data_id == 'ETTh1_Random') & (aba21.model == 'Fredformer')) |
    ((aba21.data_id == 'ETTm1_Random') & (aba21.model == 'Fredformer')) |
    ((aba21.data_id == 'ECL_Random') & (aba21.model == 'iTransformer')) |
    ((aba21.data_id == 'Weather_Random') & (aba21.model == 'FreTS'))
]
aba21 = pd.concat([aba21, df_rebb], ignore_index=True)
aba21.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba21.drop(columns=['rec_lambda'], inplace=True)



aba3 = finetunes_all.copy()
aba3 = aba3[
    (aba3.pca_dim == 'T') &
    (aba3.use_weights == 0) &
    (aba3.auxi_loss == 'MAE') &
    (aba3.reinit == 1) &
    (aba3.seq_len == 96)
]
aba3 = aba3[
    ((aba3.data_id == 'ETTh1_PCA') & (aba3.model == 'Fredformer')) |
    ((aba3.data_id == 'ETTm1_PCA') & (aba3.model == 'Fredformer')) |
    ((aba3.data_id == 'ECL_PCA') & (aba3.model == 'iTransformer')) |
    ((aba3.data_id == 'Weather_PCA') & (aba3.model == 'FreTS'))
]
aba3.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba3.drop(columns=['rec_lambda'], inplace=True)


In [ ]:
aba21.learning_rate.unique()

## analysis base

In [8]:
df1 = base.copy()

df1 = df1[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df1['data_id'] = pd.Categorical(df1['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df1['model'] = pd.Categorical(df1['model'], categories=model_order, ordered=True)

df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df1_avg['pred_len'] = 'Avg'
df1 = pd.concat([df1, df1_avg]).reset_index(drop=True)

df1.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df1.dropna(inplace=True, thresh=4)

df1['label'] = 'DF'
df1

/tmp/ipykernel_492631/1171081831.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
8,Fredformer,96,ETTm1,0.326369,0.360869,DF
9,Fredformer,192,ETTm1,0.365194,0.382132,DF
10,Fredformer,336,ETTm1,0.395987,0.404369,DF
11,Fredformer,720,ETTm1,0.459217,0.444342,DF
16,Fredformer,Avg,ETTm1,0.386692,0.397928,DF
4,Fredformer,96,ETTh1,0.377193,0.395888,DF
5,Fredformer,192,ETTh1,0.437019,0.425390,DF
6,Fredformer,336,ETTh1,0.485769,0.448580,DF
7,Fredformer,720,ETTh1,0.487772,0.467352,DF
18,Fredformer,Avg,ETTh1,0.446938,0.434302,DF


## analysis ablation 1

In [ ]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae']
# aba1[(aba1['pred_len'] == 192) & (aba1['data_id'] == 'ETTm1_PCA')].sort_values(by=['pred_len', 'mse'])[columns].round(3)
# aba1[(aba1['pred_len'] == 336) & (aba1['data_id'] == 'ETTh1_PCA')].sort_values(by=['pred_len', 'mse'])[columns].round(3)
aba1[(aba1['pred_len'] == 336) & (aba1['data_id'] == 'ECL_PCA')].sort_values(by=['pred_len', 'mse'])[columns].round(3)

In [ ]:
min_mode = 'each'

df2 = aba1.copy()
df2_m1_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTm1_PCA') & df2.index.isin([23100])]
df2_m1_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTm1_PCA') & df2.index.isin([23341])]
df2_h1_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh1_PCA') & df2.index.isin([21817])]
df2_h1_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTh1_PCA') & df2.index.isin([21979])]
df2_h1_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTh1_PCA') & df2.index.isin([22065])]
df2_ecl_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ECL_PCA') & df2.index.isin([26264])]
df2_ecl_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ECL_PCA') & df2.index.isin([27141])]
df2_ecl_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ECL_PCA') & df2.index.isin([27792])]
df2_other = df2[
    ((df2['data_id'] == 'ETTm1_PCA') & (df2['pred_len'].isin([336, 720]))) |
    ((df2['data_id'] == 'ETTh1_PCA') & (df2['pred_len'].isin([720]))) |
    ((df2['data_id'] == 'ECL_PCA') & (df2['pred_len'].isin([720]))) |
    df2.data_id.isin(['Weather_PCA'])
]
df2 = pd.concat([
    df2_m1_96, df2_m1_192, 
    df2_h1_96, df2_h1_192, df2_h1_336, 
    df2_ecl_96, df2_ecl_192, df2_ecl_336,
    df2_other
], ignore_index=True)

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    # df2 = df2.groupby(columns).filter(is_full_group)
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df2 = df2[columns]

dst_order = ['ETTm1_PCA', 'ETTm2_PCA', 'ETTh1_PCA', 'ETTh2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)
df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df2.dropna(inplace=True, thresh=5)

df2['data_id'] = df2['data_id'].str.replace('_PCA', '', regex=False)
df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df2['label'] = r'PDF$^\ddagger$'
df2

## analysis ablation 2

In [ ]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae']
aba2[(aba2['pred_len'] == 96) & (aba2['data_id'] == 'ETTm1_Random')].sort_values(by=['pred_len', 'mse'])[columns].round(3)
aba2[(aba2['pred_len'] == 336) & (aba2['data_id'] == 'Weather_Random')].sort_values(by=['pred_len', 'mse'])[columns].round(3)

In [ ]:

min_mode = 'each'

df3 = aba2.copy()
df3_m1_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'ETTm1_Random') & df3.index.isin([159])]
df3_wea_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'Weather_Random') & df3.index.isin([387])]
df3_wea_336 = df3[(df3['pred_len'] == 336) & (df3['data_id'] == 'Weather_Random') & df3.index.isin([101])]
df2_other = df3[
    ((df3['data_id'] == 'ETTm1_Random') & (df3['pred_len'].isin([192, 336, 720]))) |
    ((df3['data_id'] == 'Weather_Random') & (df3['pred_len'].isin([96, 720]))) |
    df3['data_id'].isin(['ETTh1_Random', 'ECL_Random'])
]
df3 = pd.concat([
    df3_m1_96, 
    df3_wea_192, df3_wea_336,
    df2_other
], ignore_index=True)

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df3 = df3.groupby(columns).filter(is_full_group)
    mse_mean = df3.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df3 = df3.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df3.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df3 = df3.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df3 = df3[columns]

dst_order = ['ETTm1_Random', 'ETTm2_Random', 'ETTh1_Random', 'ETTh2_Random', 'ECL_Random', 'Traffic_Random', 'Weather_Random', 'PEMS03_Random', 'PEMS08_Random']
df3['data_id'] = pd.Categorical(df3['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df3['model'] = pd.Categorical(df3['model'], categories=model_order, ordered=True)

df3_avg = df3.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df3_avg['pred_len'] = 'Avg'

df3 = pd.concat([df3, df3_avg]).reset_index(drop=True)
df3.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df3.dropna(inplace=True, thresh=5)

df3['data_id'] = df3['data_id'].str.replace('_Random', '', regex=False)
df3 = df3[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df3['label'] = r'PDF$^\dagger$'
df3

## analysis ablation 21

In [ ]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha']
# aba21[(aba21['data_id'] == 'Weather_Random')].sort_values(by=['pred_len', 'mse'])[columns].round(3)
aba21[(aba21['pred_len'] == 720) & (aba21['data_id'] == 'Weather_Random')].sort_values(by=['pred_len', 'mse'])[columns].round(5)

In [ ]:

min_mode = 'each'

df31 = aba21.copy()
df31_m1 = df31[(df31['data_id'] == 'ETTm1_Random') & df31.learning_rate.isin([0.0005])]
df31_h1 = df31[(df31['data_id'] == 'ETTh1_Random') & df31.learning_rate.isin([0.0005])]
df31_ecl = df31[(df31['data_id'] == 'ECL_Random') & df31.learning_rate.isin([0.0005])]
df31_wea_96 = df31[(df31['pred_len'] == 96) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([67])]
df31_wea_192 = df31[(df31['pred_len'] == 192) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([70])]
df31_wea_336 = df31[(df31['pred_len'] == 336) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([8])]
df31_wea_720 = df31[(df31['pred_len'] == 720) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([77])]
df31 = pd.concat([
    df31_m1, df31_h1, df31_ecl, df31_wea_96,
    df31_wea_192, df31_wea_336, df31_wea_720
], ignore_index=True)

# df31_m1_96 = df31[(df31['pred_len'] == 96) & (df31['data_id'] == 'ETTm1_Random') & df31.index.isin([159])]
# df31_wea_192 = df31[(df31['pred_len'] == 192) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([387])]
# df31_wea_336 = df31[(df31['pred_len'] == 336) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([101])]
# df31_other = df31[
#     ((df31['data_id'] == 'ETTm1_Random') & (df31['pred_len'].isin([192, 336, 720]))) |
#     ((df31['data_id'] == 'Weather_Random') & (df31['pred_len'].isin([96, 720]))) |
#     df31['data_id'].isin(['ETTh1_Random', 'ECL_Random'])
# ]
# df31 = pd.concat([
#     df31_m1_96, 
#     df31_wea_192, df31_wea_336,
#     df31_other
# ], ignore_index=True)

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df31 = df31.groupby(columns).filter(is_full_group)
    mse_mean = df31.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df31 = df31.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df31.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df31 = df31.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df31 = df31[columns]

dst_order = ['ETTm1_Random', 'ETTm2_Random', 'ETTh1_Random', 'ETTh2_Random', 'ECL_Random', 'Traffic_Random', 'Weather_Random', 'PEMS03_Random', 'PEMS08_Random']
df31['data_id'] = pd.Categorical(df31['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df31['model'] = pd.Categorical(df31['model'], categories=model_order, ordered=True)

df31_avg = df31.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df31_avg['pred_len'] = 'Avg'

df31 = pd.concat([df31, df31_avg]).reset_index(drop=True)
df31.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df31.dropna(inplace=True, thresh=5)

df31['data_id'] = df31['data_id'].str.replace('_Random', '', regex=False)
df31 = df31[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df31['label'] = r'PDF$^\dagger$_r1'
df31

## analysis ablation 3

In [ ]:
min_mode = 'each'

df4 = aba3.copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df4 = df4.groupby(columns).filter(is_full_group)
    mse_mean = df4.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df4 = df4.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df4.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df4 = df4.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df4 = df4[columns]

dst_order = ['ETTm1_PCA', 'ETTm2_PCA', 'ETTh1_PCA', 'ETTh2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']
df4['data_id'] = pd.Categorical(df4['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df4['model'] = pd.Categorical(df4['model'], categories=model_order, ordered=True)

df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df4_avg['pred_len'] = 'Avg'

df4 = pd.concat([df4, df4_avg]).reset_index(drop=True)
df4.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df4.dropna(inplace=True, thresh=5)

df4['data_id'] = df4['data_id'].str.replace('_PCA', '', regex=False)
df4 = df4[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df4['label'] = 'PDF'
df4

## concat analysis

In [ ]:
compare_columns = ['pred_len', 'mse', 'mae', 'label']
aba_show = pd.concat([df1, df3[compare_columns], df2[compare_columns], df4[compare_columns]], axis=1)


aba_res = pd.concat([df1, df3, df2, df4], axis=0)
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
aba_res.round(3)[['data_id', 'pred_len', 'mse', 'mae', 'label']].to_csv(f'{save_root}/aba_res.csv', index=False, float_format='%.3f')

dst_order = ['ETTm1', 'ETTh1', 'ECL', 'Weather']
aba_res['data_id'] = pd.Categorical(aba_res['data_id'], categories=dst_order, ordered=True)

label_order = ['DF', r'PDF$^\dagger$', r'PDF$^\ddagger$', 'PDF']
aba_res['label'] = pd.Categorical(aba_res['label'], categories=label_order, ordered=True)

aba_res.sort_values(by=['label', 'data_id', 'pred_len'], inplace=True)

aba_res = aba_res.set_index(['label', 'data_id', 'pred_len']).unstack('pred_len').swaplevel(axis=1)
columns = []
for model in aba_res.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
aba_res = aba_res[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
aba_res.round(3).to_csv(f'{save_root}/aba_res_row.csv', float_format='%.3f')

aba_res

In [ ]:
compare_columns = ['pred_len', 'mse', 'mae', 'label']
aba_show = pd.concat([df1, df31[compare_columns], df3[compare_columns], df2[compare_columns], df4[compare_columns]], axis=1)


aba_res = pd.concat([df1, df31, df3, df2, df4], axis=0)
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
# aba_res.round(3)[['data_id', 'pred_len', 'mse', 'mae', 'label']].to_csv(f'{save_root}/aba_res2.csv', index=False, float_format='%.3f')

dst_order = ['ETTm1', 'ETTh1', 'ECL', 'Weather']
aba_res['data_id'] = pd.Categorical(aba_res['data_id'], categories=dst_order, ordered=True)

label_order = ['DF', r'PDF$^\dagger$_r1', r'PDF$^\dagger$', r'PDF$^\ddagger$', 'PDF']
aba_res['label'] = pd.Categorical(aba_res['label'], categories=label_order, ordered=True)

aba_res.sort_values(by=['label', 'data_id', 'pred_len'], inplace=True)

aba_res = aba_res.set_index(['label', 'data_id', 'pred_len']).unstack('pred_len').swaplevel(axis=1)
columns = []
for model in aba_res.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
aba_res = aba_res[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
aba_res.round(3).to_csv(f'{save_root}/aba_res_row2.csv', float_format='%.3f')

aba_res